# VITON-HD Dataset Generator (Kaggle Ultimate Edition)

This notebook processes a custom dataset to generate a **complete VITON-HD compliant dataset**, including:
1. **image**: Original resized person image
2. **agnostic-v3.2**: Person image with cloth masked (gray)
3. **cloth**: Extracted clothing item
4. **cloth_mask**: Binary mask of the clothing
5. **image-parse-v3**: Human parsing labels
6. **image-parse-agnostic-v3.2**: Parsing with cloth region masked
7. **openpose_img**: OpenPose keypoint visualization
8. **openpose_json**: OpenPose keypoints JSON
9. **densepose**: DensePose IUV mapping

**Hardware**: Requires **GPU T4 x 2** or **P100**.

## 1. Install Dependencies

In [ ]:
print("Installing dependencies... (This may take a few minutes)")
!pip install -q segment-anything transformers
!pip install -q opencv-python-headless matplotlib tqdm gdown scipy pycocotools
!pip install -q controlnet_aux==0.0.6
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# Clean clone verify
import os
import shutil
if os.path.exists("detectron2"):
    shutil.rmtree("detectron2")

print("Cloning detectron2 to get configs...")
!git clone https://github.com/facebookresearch/detectron2.git
print("Dependencies installed.")

In [ ]:
!pip install -U pip
!pip install av
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

#restart kernel

## 2. Models & Configuration

In [ ]:
import os
import torch
import numpy as np
import cv2
import gdown
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json

# Create directories
os.makedirs("models", exist_ok=True)

# Download SAM
sam_path = "models/sam_vit_h_4b8939.pth"
if not os.path.exists(sam_path):
    gdown.download("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth", sam_path, quiet=False, fuzzy=True)

# Config
INPUT_DIR = Path("/kaggle/input/sample-kamiz") 
OUTPUT_DIR = Path("/kaggle/working/output")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = (768, 1024) # H, W (Wait, PIL resize is W, H)
# PIL resize takes (Width, Height). Standard VITON-HD is 768x1024 (WxH).
PIL_SIZE = (768, 1024)

print(f"Device: {DEVICE}")

## 3. Generators (Parsing, OpenPose, DensePose)

In [ ]:
import torch.nn as nn
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
from segment_anything import sam_model_registry, SamPredictor
from controlnet_aux import OpenposeDetector
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2 import model_zoo
import sys, os

# --- Human Parser (Segformer) ---
class HumanParser:
    def __init__(self, device=DEVICE):
        self.device = device
        print("Loading Segformer...")
        self.processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
        self.model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes")
        self.model.to(self.device)
    
    def __call__(self, image): # image is numpy RGB
        inputs = self.processor(images=image, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits.cpu()
        upsampled = nn.functional.interpolate(logits, size=image.shape[:2], mode="bilinear", align_corners=False)
        pred = upsampled.argmax(dim=1)[0].numpy().astype(np.uint8)
        return self._map_to_lip(pred)

    def _map_to_lip(self, seg):
        mapping = {0:0, 1:1, 2:2, 3:4, 4:5, 5:12, 6:9, 7:6, 8:0, 9:18, 10:19, 11:13, 12:16, 13:17, 14:14, 15:15, 16:0, 17:11}
        lip = np.zeros_like(seg)
        for s, l in mapping.items(): lip[seg==s] = l
        return lip

    def colorize(self, parsing):
        h, w = parsing.shape
        vis = np.zeros((h, w, 3), dtype=np.uint8)
        palette = {
            0:  [0, 0, 0],       # Background
            1:  [255, 0, 0],     # Hat
            2:  [255, 0, 0],     # Hair
            5:  [255, 85, 0],    # Upper
            6:  [255, 85, 0],    # Dress
            7:  [255, 85, 0],    # Coat
            11: [255, 85, 0],    # Scarf/Dupatta
            12: [255, 85, 0],    # Skirt
            9:  [0, 85, 85],     # Pants
            10: [0, 85, 85],     # Skirt
            13: [0, 0, 255],     # Face
            14: [85, 255, 255],  # L-Arm
            15: [85, 255, 255],  # R-Arm
            16: [0, 128, 128],   # L-Leg
            17: [0, 128, 128],   # R-Leg
            18: [255, 255, 255], # L-Shoe
            19: [255, 255, 255], # R-Shoe
        }
        for label, color in palette.items():
            vis[parsing == label] = color
        return vis

    def get_cloth_mask(self, parsing):
        """Get cloth mask for kamiz with dupatta.
        Includes everything that's NOT background, head, or lower body.
        This captures dupatta even if not detected as 'scarf' by Segformer.
        """
        mask = np.zeros_like(parsing, dtype=np.uint8)
        # Everything that's NOT in these labels is considered cloth
        non_cloth_labels = [0, 1, 2, 13, 9, 16, 17, 18, 19]  # bg, hat, hair, face, pants, legs, shoes
        cloth_region = ~np.isin(parsing, non_cloth_labels)
        mask[cloth_region] = 255
        return mask

# --- OpenPose Generator ---
class OpenPoseGen:
    def __init__(self, device=DEVICE):
        print("Loading OpenPose...")
        self.model = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
        self.model.to(device)
    
    def __call__(self, image_pil):
        pose_img = self.model(image_pil)
        img_np = np.array(image_pil)
        candidate, subset = self.model.body_estimation(img_np)
        keypoints = []
        if subset.shape[0] > 0:
            person = subset[0][:18]
            for i in range(18):
                index = int(person[i])
                if index == -1: keypoints.extend([0, 0, 0])
                else:
                    x, y = candidate[index][0:2]
                    keypoints.extend([float(x), float(y), 1.0])
        pose_json = {"version": 1.0, "people": [{"pose_keypoints_2d": keypoints}]}
        return pose_img, pose_json

# --- DensePose Generator ---
class DensePoseGen:
    def __init__(self, device=DEVICE):
        print("Loading DensePose...")
        densepose_dir = os.path.join(os.getcwd(), "detectron2", "projects", "DensePose")
        if densepose_dir not in sys.path: sys.path.insert(0, densepose_dir)
        from densepose import add_densepose_config
        cfg = get_cfg()
        add_densepose_config(cfg)
        cfg.merge_from_file("detectron2/projects/DensePose/configs/densepose_rcnn_R_50_FPN_s1x.yaml")
        cfg.MODEL.WEIGHTS = "https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl"
        cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
        self.predictor = DefaultPredictor(cfg)

    def __call__(self, image_np, body_mask=None):
        """Generate DensePose. If body_mask provided, use it to constrain output to body silhouette."""
        img_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
        with torch.no_grad(): 
            outputs = self.predictor(img_bgr)["instances"]
        
        if not outputs.has("pred_densepose") or len(outputs) == 0: 
            return np.zeros_like(image_np)
            
        # Pick the largest detection (main person)
        best_idx = torch.argmax(outputs.pred_boxes.area()).item()
        results = outputs.pred_densepose
        i_logits = results.fine_segm[best_idx]
        
        # Get body part indices
        i_indices = torch.argmax(i_logits, dim=0).cpu().numpy().astype(np.uint8)
        
        # Clean up with median filter
        i_indices = cv2.medianBlur(i_indices, 5)
        
        h_dp, w_dp = i_indices.shape
        
        # DensePose colormap matching reference
        palette = {
            0: [0, 0, 0],
            1: [0, 85, 255], 2: [0, 85, 255],  # Torso - Blue
            3: [0, 255, 127], 4: [0, 255, 127], 5: [0, 255, 127], 6: [0, 255, 127],  # Hands/Feet
            7: [0, 255, 255], 8: [0, 255, 255], 9: [0, 255, 255], 10: [0, 255, 255],  # Legs - Cyan
            11: [0, 170, 255], 12: [0, 170, 255], 13: [0, 170, 255], 14: [0, 170, 255],
            15: [170, 255, 85], 16: [170, 255, 85], 17: [170, 255, 85], 18: [170, 255, 85],  # Arms
            19: [85, 255, 170], 20: [85, 255, 170], 21: [85, 255, 170], 22: [85, 255, 170],
            23: [255, 255, 0], 24: [255, 255, 0],  # Head - Yellow
        }
        
        vis_low = np.zeros((h_dp, w_dp, 3), dtype=np.uint8)
        for idx, col in palette.items():
            vis_low[i_indices == idx] = col
        
        # Get bounding box
        box = outputs.pred_boxes.tensor[best_idx].cpu().numpy().astype(int)
        x, y, x2, y2 = box
        w, h = x2-x, y2-y
        if w <= 0 or h <= 0: return np.zeros_like(image_np)
        
        # Resize to bounding box size
        vis_patch = cv2.resize(vis_low, (w, h), interpolation=cv2.INTER_NEAREST)
        
        # Create full output
        full_vis = np.zeros_like(image_np)
        y_end, x_end = min(y+h, image_np.shape[0]), min(x+w, image_np.shape[1])
        h_real, w_real = y_end-y, x_end-x
        
        if h_real > 0 and w_real > 0:
            full_vis[y:y_end, x:x_end] = vis_patch[:h_real, :w_real]
        
        # CRITICAL: Apply body mask to REMOVE all scattered patches outside person silhouette
        if body_mask is not None:
            # Dilate mask slightly for complete coverage
            kernel = np.ones((5, 5), np.uint8)
            dilated_mask = cv2.dilate(body_mask.astype(np.uint8), kernel, iterations=2)
            # Zero out everything outside the body
            full_vis[dilated_mask == 0] = 0
        
        return full_vis

class ClothSegmenter:
    def __init__(self, sam_path, device=DEVICE):
        sam = sam_model_registry["vit_h"](checkpoint=sam_path).to(device=device)
        self.predictor = SamPredictor(sam)
    def __call__(self, image, parsing_mask):
        coords = np.column_stack(np.where(parsing_mask > 0))
        if len(coords) == 0: return image, parsing_mask
        y1, x1 = coords.min(axis=0)
        y2, x2 = coords.max(axis=0)
        self.predictor.set_image(image)
        masks, _, _ = self.predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=False)
        mask = (masks[0] * 255).astype(np.uint8)
        mask = cv2.bitwise_and(mask, parsing_mask)
        cloth = image.copy()
        cloth[mask==0] = 255
        return cloth, mask

# --- Agnostic Generator ---
class AgnosticGen:
    def __call__(self, image, parsing, cloth_mask=None):
        """Create agnostic image for kamiz.
        Masks entire upper body gray, keeps only head, hands, and lower body.
        """
        agnostic = image.copy()
        h, w = parsing.shape
        
        # LIP Labels to KEEP visible (not mask gray):
        # 0: Background
        # 1: Hat, 2: Hair
        # 13: Face
        # 9: Pants
        # 16, 17: Left/Right Leg
        # 18, 19: Left/Right Shoe
        # Note: We intentionally DON'T keep arms (14, 15) since they're part of upper body
        keep_labels = [0, 1, 2, 13, 9, 16, 17, 18, 19]
        
        # Create mask of regions to KEEP
        keep_mask = np.isin(parsing, keep_labels)
        
        # Mask everything else gray (this includes all upper body + dupatta area)
        agnostic[~keep_mask] = [128, 128, 128]
        
        return agnostic
    
    def get_parse_agnostic(self, parsing, cloth_mask=None):
        """Create agnostic parsing - zero out everything except head and lower body."""
        p_agnostic = parsing.copy()
        keep_labels = [0, 1, 2, 13, 9, 16, 17, 18, 19]
        p_agnostic[~np.isin(p_agnostic, keep_labels)] = 0
        return p_agnostic



## 4. Main Pipeline

In [ ]:
def run_pipeline():
    # Initialize
    parser = HumanParser()
    openpose = OpenPoseGen()
    densepose = DensePoseGen()
    segmenter = ClothSegmenter(sam_path)
    agnostic_gen = AgnosticGen()
    
    # Dirs
    subdirs = ["image", "agnostic-v3.2", "cloth", "cloth_mask", "image-parse-v3", "image-parse-agnostic-v3.2", "openpose_img", "openpose_json", "densepose"]
    for s in subdirs: (OUTPUT_DIR / s).mkdir(parents=True, exist_ok=True)
    
    images = list(INPUT_DIR.rglob("*.jpg")) + list(INPUT_DIR.rglob("*.png"))
    print(f"Found {len(images)} images")
    
    for path in tqdm(images):
        try:
            name = f"{path.parent.parent.name}_{path.parent.name}_{path.name}"
            img_pil = Image.open(path).convert("RGB").resize(PIL_SIZE, Image.LANCZOS)
            img_np = np.array(img_pil)
            
            parsing = parser(img_np)
            cloth_mask_p = parser.get_cloth_mask(parsing)
            cloth, cloth_mask = segmenter(img_np, cloth_mask_p)
            agnostic = agnostic_gen(img_np, parsing, cloth_mask)
            parse_agnostic = agnostic_gen.get_parse_agnostic(parsing, cloth_mask)
            pose_img, pose_json = openpose(img_pil)
            
            # Create body mask from parsing (all non-zero labels = body)
            body_mask = (parsing > 0).astype(np.uint8) * 255
            # Pass body_mask to densepose to remove scattered patches
            dense_img = densepose(img_np, body_mask)
            
            # SAVE
            Image.fromarray(img_np).save(OUTPUT_DIR / "image" / f"{name}.jpg")
            Image.fromarray(agnostic).save(OUTPUT_DIR / "agnostic-v3.2" / f"{name}.jpg")
            Image.fromarray(cloth).save(OUTPUT_DIR / "cloth" / f"{name}.jpg")
            Image.fromarray(cloth_mask).save(OUTPUT_DIR / "cloth_mask" / f"{name}.jpg")
            Image.fromarray(parser.colorize(parsing)).save(OUTPUT_DIR / "image-parse-v3" / f"{name}.png")
            Image.fromarray(parser.colorize(parse_agnostic)).save(OUTPUT_DIR / "image-parse-agnostic-v3.2" / f"{name}.png")
            pose_img.save(OUTPUT_DIR / "openpose_img" / f"{name}.jpg")
            Image.fromarray(dense_img).save(OUTPUT_DIR / "densepose" / f"{name}.jpg")
            with open(OUTPUT_DIR / "openpose_json" / f"{name}.json", 'w') as f: json.dump(pose_json, f)
                
        except Exception as e:
            print(f"Err {path.name}: {e}")
            
    print("Done. Zipping...")
    !zip -q -r viton_hd_output.zip output
    print("Zip created.")

if __name__ == "__main__":
    run_pipeline()

